# Khảo sát MovieLens bằng Spark
Đọc CSV gốc và ghi thống kê thực tế. Chọn DATASET phù hợp; không dùng số mẫu viết sẵn làm kết quả. Source lấy từ GitHub theo SOURCE_REF.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, json, shutil, subprocess, zipfile, time
from collections import deque

BASE = Path('/content/drive/MyDrive/BigData/DoAn_GoiYPhim_Alluxio')
PROJECT = Path('/content/movie-recommendation-alluxio')
REPOSITORY = 'https://github.com/iuzenaurora/movie-recommendation-alluxio.git'
SOURCE_REF = 'origin/main'  # Use a commit SHA for a reproducible experiment.
for name in ['01_Dataset_Raw','02_Dataset_Processed','03_Notebooks','04_Models','05_Experiments','06_Documents']:
    (BASE / name).mkdir(parents=True, exist_ok=True)
if not PROJECT.exists():
    subprocess.run(['git','clone',REPOSITORY,str(PROJECT)],check=True)
if not (PROJECT / '.git').is_dir():
    raise ValueError('PROJECT already exists without Git; select a new project directory.')
if subprocess.check_output(['git','-C',str(PROJECT),'status','--porcelain'],text=True).strip():
    raise ValueError('Source has local changes. Preserve them before selecting another revision.')
subprocess.run(['git','-C',str(PROJECT),'fetch','origin'],check=True)
subprocess.run(['git','-C',str(PROJECT),'checkout','--detach',SOURCE_REF],check=True)
os.chdir(PROJECT)
java_home = Path('/usr/lib/jvm/java-17-openjdk-amd64')
if not java_home.exists():
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','-qq','openjdk-17-jre-headless'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','pyspark==3.5.6','numpy','matplotlib'],check=True)
os.environ['JAVA_HOME'] = str(java_home)
os.environ['PATH'] = str(java_home / 'bin') + ':' + os.environ['PATH']
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYTHONPATH'] = str(PROJECT / 'src')
sys.path.insert(0,str(PROJECT / 'src'))
import numpy as np
print('Source commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
print('Python:',sys.version,'NumPy:',np.__version__)
subprocess.run(['java','-version'],check=True)

def extract_checked(path, destination):
    with zipfile.ZipFile(path) as archive:
        if archive.testzip() is not None:
            raise ValueError('Corrupt ZIP: ' + str(path))
        for name in archive.namelist():
            if not (destination / name).resolve().is_relative_to(destination.resolve()):
                raise ValueError('Unsafe ZIP path')
        archive.extractall(destination)


In [ ]:
from pyspark.sql import SparkSession, functions as F
DATASET = 'ml-latest-small'  # Change to ml-32m after downloading that dataset.
RAW = BASE / '01_Dataset_Raw' / DATASET
assert (RAW / 'ratings.csv').is_file() and (RAW / 'movies.csv').is_file(), 'Download raw data using notebook 00 first.'
spark = SparkSession.builder.master('local[2]').appName('MovieLens-EDA').getOrCreate()
ratings = spark.read.option('header',True).schema('userId INT,movieId INT,rating DOUBLE,timestamp LONG').csv(str(RAW / 'ratings.csv'))
movies = spark.read.option('header',True).schema('movieId INT,title STRING,genres STRING').csv(str(RAW / 'movies.csv'))
stats = ratings.agg(F.count('*').alias('ratings'),F.countDistinct('userId').alias('users'),F.countDistinct('movieId').alias('moviesRated'),F.min('rating').alias('minRating'),F.max('rating').alias('maxRating')).first().asDict()
stats['moviesCatalog'] = movies.count()
stats['invalidRows'] = ratings.filter(F.col('userId').isNull() | F.col('movieId').isNull() | F.col('rating').isNull() | F.col('timestamp').isNull() | (F.col('userId')<=0) | (F.col('movieId')<=0) | (F.col('timestamp')<0) | ~F.col('rating').between(.5,5) | ((F.col('rating')*2)%1!=0)).count()
stats['duplicateUserMoviePairs'] = ratings.groupBy('userId','movieId').count().filter('count > 1').count()
print(json.dumps(stats,indent=2))
ratings.groupBy('rating').count().orderBy('rating').show()
movies.select(F.explode(F.split('genres',r'\|')).alias('genre')).groupBy('genre').count().orderBy(F.desc('count')).show(30)
report = BASE / '05_Experiments' / (DATASET+'-eda-'+time.strftime('%Y%m%d-%H%M%S')+'.json')
report.write_text(json.dumps(stats,indent=2),encoding='utf-8')
spark.stop()
print('Saved:',report)
